In [ ]:
from pyspark.sql import SparkSession
import os

spark = (
    SparkSession.builder.appName("iceberg")
    .master("local[*]")
    .config("spark.driver.memory", "8g")
    .config(
        "spark.jars.packages",
        "org.apache.iceberg:iceberg-spark-runtime-4.0_2.13:1.10.1,software.amazon.awssdk:bundle:2.20.162",
    )
    .config(
        "spark.sql.extensions",
        "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions",
    )
    .config("spark.sql.catalog.rest", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.rest.type", "rest")
    .config("spark.sql.catalog.rest.uri", os.environ["ICEBERG_REST_URI"])
    .config("spark.sql.catalog.rest.io-impl", "org.apache.iceberg.aws.s3.S3FileIO")
    .config("spark.sql.catalog.rest.s3.endpoint", os.environ["ICEBERG_S3_ENDPOINT_PUBLIC"])
    .config("spark.sql.catalog.rest.s3.path-style-access", "true")
    .config("spark.sql.catalog.rest.s3.access-key-id", os.environ["ICEBERG_S3_ACCESS_KEY_ID"])
    .config(
        "spark.sql.catalog.rest.s3.secret-access-key",
        os.environ["ICEBERG_S3_SECRET_ACCESS_KEY"],
    )
    .getOrCreate()
)


:: loading settings :: url = jar:file:/mnt/disk1/project/project/aikb/bio_info/.venv/lib/python3.13/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/wjx/.ivy2.5.2/cache
The jars for the packages stored in: /home/wjx/.ivy2.5.2/jars
org.apache.iceberg#iceberg-spark-runtime-4.0_2.13 added as a dependency
software.amazon.awssdk#bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-d57033e4-f4cd-4068-b829-81e7d4f9371c;1.0
	confs: [default]
	found org.apache.iceberg#iceberg-spark-runtime-4.0_2.13;1.10.1 in central
	found software.amazon.awssdk#bundle;2.20.162 in central
	found software.amazon.eventstream#eventstream;1.0.1 in central
downloading https://repo1.maven.org/maven2/software/amazon/awssdk/bundle/2.20.162/bundle-2.20.162.jar ...
	[SUCCESSFUL ] software.amazon.awssdk#bundle;2.20.162!bundle.jar (389450ms)
:: resolution report :: resolve 2732ms :: artifacts dl 389453ms
	:: modu

In [2]:
from pyiceberg.catalog import load_catalog
from pyiceberg.schema import Schema
from pyiceberg.types import LongType, StringType, DoubleType

catalog = load_catalog(
    "rest",
    **{
        "type": "rest",
        "uri": os.environ["ICEBERG_REST_URI"],
        "s3.endpoint": os.environ["ICEBERG_S3_ENDPOINT_PUBLIC"],
        "s3.access-key-id": os.environ["ICEBERG_S3_ACCESS_KEY_ID"],
        "s3.secret-access-key": os.environ["ICEBERG_S3_SECRET_ACCESS_KEY"],
    },
)

# 创建命名空间
# catalog.create_namespace("ukb")

In [3]:
# List existed namespaces
catalog.list_namespaces()

[('ukb',)]

In [4]:
# Load dataset by pyspark
olink_df = spark.read.parquet("/mnt/disk2/dataset/ukb/olink0_data.parquet", header=True)
hp_df = spark.read.parquet("/mnt/disk2/dataset/ukb/hypertension.parquet", header=True)

In [5]:
olink_df.limit(10).toPandas()

26/04/15 14:19:18 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


,eid,a1bg,aamdc,aarsd1,abca2,abhd14b,abl1,abo,abraxas2,acaa1,...,zfyve19,zhx2,znf174,znf75d,znf830,znrd2,znrf4,zp3,zp4,zpr1
0,1447930,-0.0023,-0.08435,-0.92140,0.16615,-0.12760,-0.10010,0.3390,-1.1846,-0.66010,...,0.2541,-0.0897,-0.0784,2.2171,-0.11460,-0.00565,-0.7039,-5.6609,0.9599,-0.1789
1,1447992,0.1510,-0.51985,0.19385,0.44080,-0.25490,-0.07300,-2.1080,-0.7779,NaN,...,-0.6637,0.0408,-0.2789,NaN,-0.44160,0.39765,-0.2825,0.8887,-0.0666,0.0008
2,1448204,NaN,0.75245,-0.18485,0.04760,0.42155,0.18830,NaN,0.1806,0.70435,...,0.0271,0.0983,0.4701,NaN,-0.13190,-0.92105,-0.5728,1.1031,-0.5997,0.5458
3,1448381,0.1111,0.33875,0.31640,0.78215,1.05750,1.90600,-1.8991,0.9604,0.69730,...,2.0509,0.2423,0.0042,0.0485,2.57350,0.67485,1.5133,0.7254,0.0248,1.0470
4,1448954,NaN,NaN,0.04295,NaN,-0.22255,-0.33975,NaN,NaN,-0.45080,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,1449036,-0.2141,-0.11355,0.58255,-0.07280,0.29815,0.40280,0.6321,0.2901,0.51695,...,0.0581,0.4536,0.0212,-0.2044,1.09400,0.41315,0.2806,1.7806,-0.0452,0.4077
6,1449411,-0.0162,-0.27725,0.69525,-0.47520,-0.36710,-0.80330,-0.8963,-0.5932,-0.51120,...,-1.4086,0.1111,0.2761,-0.0844,0.52470,-0.59435,-0.7134,-1.0327,-0.0981,-0.3213
7,1449929,-0.0787,NaN,-0.20480,0.23375,-0.32540,-0.60690,-0.3124,-0.8368,-0.12930,...,-0.7159,0.0390,-0.5844,-0.1664,-0.41220,-0.03945,-0.2943,NaN,-0.0354,-0.0360
8,1450187,0.1464,0.35645,1.72760,0.28160,0.93680,1.69900,2.3618,0.1083,1.31030,...,-0.0827,4.6262,0.8941,0.2382,1.71525,0.93120,-0.2803,-5.0456,-0.2495,-0.6382
9,1451620,NaN,NaN,0.01485,NaN,0.33365,0.56625,NaN,NaN,0.71190,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
hp_df.limit(10).toPandas()

,Unnamed: 0,participant.eid,participant.p131286,participant.p21003_i0,participant.p21001_i0,participant.p48_i0,participant.p1239_i0,participant.p1249_i0,participant.p2976_i0,olink_instance_0.eid
0,0,1003370,2013-07-01,66,29.957936,109.0,0.0,1.0,NaN,1003370
1,1,1003757,2014-08-14,46,45.105601,108.0,0.0,4.0,NaN,1003757
2,2,1004168,2013-04-01,56,26.367087,93.0,0.0,1.0,NaN,1004168
3,3,1004587,2001-09-01,59,33.184348,101.0,0.0,1.0,NaN,1004587
4,4,1007786,2008-09-01,59,43.510937,117.0,0.0,4.0,NaN,1007786
5,5,1007995,1998-08-12,55,25.238535,87.0,0.0,3.0,NaN,1007995
6,6,1011323,2000-11-01,62,36.884288,106.0,0.0,1.0,NaN,1011323
7,7,1014157,2007-11-01,66,30.590860,101.0,0.0,4.0,NaN,1014157
8,8,1016393,1994-05-01,65,24.454086,87.0,0.0,3.0,NaN,1016393
9,9,1016864,2007-02-01,60,39.697543,124.0,1.0,NaN,57.0,1016864


In [7]:
hp_df.printSchema()

root
 |-- Unnamed: 0: long (nullable = true)
 |-- participant.eid: long (nullable = true)
 |-- participant.p131286: string (nullable = true)
 |-- participant.p21003_i0: long (nullable = true)
 |-- participant.p21001_i0: double (nullable = true)
 |-- participant.p48_i0: double (nullable = true)
 |-- participant.p1239_i0: double (nullable = true)
 |-- participant.p1249_i0: double (nullable = true)
 |-- participant.p2976_i0: double (nullable = true)
 |-- olink_instance_0.eid: long (nullable = true)



In [8]:
from functools import reduce

hp_df = reduce(
    lambda df, c: df.withColumnRenamed(c, c.replace(".", "_")),
    hp_df.columns,
    hp_df,
)
hp_df.describe().show()


+-------+-----------------+-----------------+-------------------+---------------------+---------------------+------------------+--------------------+--------------------+--------------------+--------------------+
|summary|       Unnamed: 0|  participant_eid|participant_p131286|participant_p21003_i0|participant_p21001_i0|participant_p48_i0|participant_p1239_i0|participant_p1249_i0|participant_p2976_i0|olink_instance_0_eid|
+-------+-----------------+-----------------+-------------------+---------------------+---------------------+------------------+--------------------+--------------------+--------------------+--------------------+
|  count|            22987|            22987|              22987|                22987|                22838|             22905|               22960|               21123|                2424|               22987|
|   mean|          11493.0|3513286.937312394|               NULL|   59.604385087223214|   28.956065536529877| 95.11633267845438| 0.12791811846689896

In [ ]:
olink_df.writeTo("rest.ukb.olink_instance_0").create()

SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.
26/04/15 14:19:30 ERROR Executor: Exception in task 3.0 in stage 8.0 (TID 10)12]
software.amazon.awssdk.core.exception.SdkClientException: Unable to load region from any of the providers in the chain software.amazon.awssdk.regions.providers.DefaultAwsRegionProviderChain@43e61f2c: [software.amazon.awssdk.regions.providers.SystemSettingsRegionProvider@3b02f1d9: Unable to load region from system settings. Region must be specified either via environment variable (AWS_REGION) or  system property (aws.region)., software.amazon.awssdk.regions.providers.AwsProfileRegionProvider@314260c8: No region provided in profile: default, software.amazon.awssdk.regions.providers.InstanceProfileRegionProvider@19915a12: Unable to contact EC2 metadata service.]
	at software.amazon.awssdk.core.exception

Py4JJavaError: An error occurred while calling o87.create.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 3 in stage 8.0 failed 1 times, most recent failure: Lost task 3.0 in stage 8.0 (TID 10) (archlinux executor driver): software.amazon.awssdk.core.exception.SdkClientException: Unable to load region from any of the providers in the chain software.amazon.awssdk.regions.providers.DefaultAwsRegionProviderChain@43e61f2c: [software.amazon.awssdk.regions.providers.SystemSettingsRegionProvider@3b02f1d9: Unable to load region from system settings. Region must be specified either via environment variable (AWS_REGION) or  system property (aws.region)., software.amazon.awssdk.regions.providers.AwsProfileRegionProvider@314260c8: No region provided in profile: default, software.amazon.awssdk.regions.providers.InstanceProfileRegionProvider@19915a12: Unable to contact EC2 metadata service.]
	at software.amazon.awssdk.core.exception.SdkClientException$BuilderImpl.build(Sdk

26/04/15 14:19:36 WARN TaskSetManager: Lost task 0.0 in stage 8.0 (TID 7) (archlinux executor driver): TaskKilled (Stage cancelled: Job aborted due to stage failure: Task 3 in stage 8.0 failed 1 times, most recent failure: Lost task 3.0 in stage 8.0 (TID 10) (archlinux executor driver): software.amazon.awssdk.core.exception.SdkClientException: Unable to load region from any of the providers in the chain software.amazon.awssdk.regions.providers.DefaultAwsRegionProviderChain@43e61f2c: [software.amazon.awssdk.regions.providers.SystemSettingsRegionProvider@3b02f1d9: Unable to load region from system settings. Region must be specified either via environment variable (AWS_REGION) or  system property (aws.region)., software.amazon.awssdk.regions.providers.AwsProfileRegionProvider@314260c8: No region provided in profile: default, software.amazon.awssdk.regions.providers.InstanceProfileRegionProvider@19915a12: Unable to contact EC2 metadata service.]
	at software.amazon.awssdk.core.exception.Sd